In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

plt.rcParams["figure.figsize"] = (20, 13)
%matplotlib inline
%config InlineBackend.figure_format = "retina"

In [7]:
interactions = pd.read_csv("./data_final_project/KuaiRec/data/big_matrix.csv")
small_interactions = pd.read_csv("./data_final_project/KuaiRec/data/small_matrix.csv")
captions = pd.read_csv("./data_final_project/KuaiRec/data/kuairec_caption_category.csv", lineterminator='\n')

def clean_df(df):
    df = df.dropna()
    df = df.drop_duplicates()  
    return df  

def clean_df_timestamp(df):
    df = clean_df(df)
    df = df[df["timestamp"] >= 0]
    return df

captions = clean_df(captions)
captions = captions.drop_duplicates(subset='video_id')
train_df = clean_df_timestamp(interactions)
test_df = clean_df_timestamp(small_interactions)

In [8]:
captions.columns

Index(['video_id', 'manual_cover_text', 'caption', 'topic_tag',
       'first_level_category_id', 'first_level_category_name',
       'second_level_category_id', 'second_level_category_name',
       'third_level_category_id', 'third_level_category_name'],
      dtype='object')

In [9]:
captions.head()

,video_id,manual_cover_text,caption,topic_tag,first_level_category_id,first_level_category_name,second_level_category_id,second_level_category_name,third_level_category_id,third_level_category_name
0,0,UNKNOWN,精神小伙路难走 程哥你狗粮慢点撒,[],8,颜值,673,颜值随拍,-124,UNKNOWN
2,2,UNKNOWN,晚饭后，运动一下！,[],9,喜剧,727,搞笑互动,-124,UNKNOWN
3,3,UNKNOWN,我平淡无奇，惊艳不了时光，温柔不了岁月，我只想漫无目的的走走，努力发笔小财，给自己买花 自己长大.,[],26,摄影,686,主题摄影,2434,景物摄影
4,4,五爱街最美美女 一天1q,#搞笑 #感谢快手我要上热门 #五爱市场 这真是完美搭配啊！,"[五爱市场,感谢快手我要上热门,搞笑]",5,时尚,737,营销售卖,2596,女装
5,5,UNKNOWN,“你们吵的越狠 他们的手就握的越紧” #文轩 #刘耀文 #宋亚轩 #顾子璇...,"[刘耀文,宋亚轩,文轩,顾子璇是樱桃吖,顾子璇超级喜欢文轩]",6,明星娱乐,667,娱乐八卦,2375,饭制


In [10]:
%%bash
pip install jieba

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━�m╺�━ 0.0/19.2 MB ? eta -:--:--��━━━━━━━━━━━━━━━━━ 4.2/19.2 MB 22.9 MB/s eta 0:00:01��━━━━━━━━━━━━━━━━━ 10.2/19.2 MB 26.2 MB/s eta 0:00:01��━━━━╺━━━━━━ 16.0/19.2 MB 27.0 MB/s eta 0:00:01��━━━━━━━━━━━ 19.2/19.2 MB 25.3 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for jieba: filename=jieba-0.42.1-py3-none-any.whl size=19314508 sha256=828e221e5789e7cf64082004b29d50b62f2c4a5165c0f0fa13dff1e80e5aedf7
  Stored in directory: /Users/maxboc/Library/Caches/pip/wheels/08/a1/a3/5c8ac52cc2f5782ffffc34c95c57c8e5ecb3063dc69541ee7c
Successfully built jieba


In [12]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from sklearn.feature_extraction.text import TfidfVectorizer

# Load a pretrained tokenizer or train your own
# For example: load a pretrained Chinese tokenizer (like BERT or a BPE model)
tokenizer = Tokenizer.from_pretrained("bert-base-chinese")  # if available

# Tokenization function using the tokenizer
def tokenize_bpe(text):
    return ' '.join(tokenizer.encode(text).tokens)

# Apply BPE tokenization
captions['tokenized'] = captions['caption'].apply(tokenize_bpe)

# TF-IDF on BPE tokens
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(captions['tokenized'])

print(tfidf_matrix.shape)


tokenizer.json:   0%|          | 0.00/269k [00:00<?, ?B/s]

(9373, 458)


In [14]:
captions['tokenized']

0                [CLS] 精 神 小 伙 路 难 走 程 哥 你 狗 粮 慢 点 撒 [SEP]
2                            [CLS] 晚 饭 后 ， 运 动 一 下 ！ [SEP]
3        [CLS] 我 平 淡 无 奇 ， 惊 艳 不 了 时 光 ， 温 柔 不 了 岁 月 ， ...
4        [CLS] # 搞 笑 # 感 谢 快 手 我 要 上 热 门 # 五 爱 市 场 这 真 ...
5        [CLS] [UNK] 你 们 吵 的 越 狠 他 们 的 手 就 握 的 越 紧 [UNK...
                               ...                        
10722    [CLS] # 2020 新 款 # 民 族 复 古 风 # 原 创 视 频 # 作 品 推...
10723    [CLS] 昨 天 爱 你 ， 今 天 爱 你 ， 明 天 也 爱 你 ， 丫 头 ， 别 ...
10724      [CLS] # 感 谢 推 广 小 助 手 # 感 谢 快 手 绿 色 平 台 # [SEP]
10726    [CLS] 老 人 言 ， 喜 欢 留 个 关 注 加 红 心 # 老 人 言 @ 今 天 ...
10727    [CLS] 一 眼 就 喜 欢 的 链 条 纯 色 卫 衣 百 搭 颜 色 黑 白 灰 喜 ...
Name: tokenized, Length: 9373, dtype: object

In [16]:
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

In [17]:
indices = pd.Series(captions.index, index=captions['video_id']).drop_duplicates()

In [18]:
captions.head()

,video_id,manual_cover_text,caption,topic_tag,first_level_category_id,first_level_category_name,second_level_category_id,second_level_category_name,third_level_category_id,third_level_category_name,tokenized
0,0,UNKNOWN,精神小伙路难走 程哥你狗粮慢点撒,[],8,颜值,673,颜值随拍,-124,UNKNOWN,[CLS] 精 神 小 伙 路 难 走 程 哥 你 狗 粮 慢 点 撒 [SEP]
2,2,UNKNOWN,晚饭后，运动一下！,[],9,喜剧,727,搞笑互动,-124,UNKNOWN,[CLS] 晚 饭 后 ， 运 动 一 下 ！ [SEP]
3,3,UNKNOWN,我平淡无奇，惊艳不了时光，温柔不了岁月，我只想漫无目的的走走，努力发笔小财，给自己买花 自己长大.,[],26,摄影,686,主题摄影,2434,景物摄影,[CLS] 我 平 淡 无 奇 ， 惊 艳 不 了 时 光 ， 温 柔 不 了 岁 月 ， ...
4,4,五爱街最美美女 一天1q,#搞笑 #感谢快手我要上热门 #五爱市场 这真是完美搭配啊！,"[五爱市场,感谢快手我要上热门,搞笑]",5,时尚,737,营销售卖,2596,女装,[CLS] # 搞 笑 # 感 谢 快 手 我 要 上 热 门 # 五 爱 市 场 这 真 ...
5,5,UNKNOWN,“你们吵的越狠 他们的手就握的越紧” #文轩 #刘耀文 #宋亚轩 #顾子璇...,"[刘耀文,宋亚轩,文轩,顾子璇是樱桃吖,顾子璇超级喜欢文轩]",6,明星娱乐,667,娱乐八卦,2375,饭制,[CLS] [UNK] 你 们 吵 的 越 狠 他 们 的 手 就 握 的 越 紧 [UNK...


In [19]:
def get_recommendations(title, cosine_sim=cosine_sim, num_recommend = 10):
    idx = indices[title]
    # Get the pairwsie similarity scores of all movies with that movie
    sim_scores = list(enumerate(cosine_sim[idx]))
    # Sort the movies based on the similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    # Get the scores of the 10 most similar movies
    top_similar = sim_scores[1:num_recommend+1]
    # Get the movie indices
    movie_indices = [i[0] for i in top_similar]
    # Return the top 10 most similar movies
    return captions['video_id'].iloc[movie_indices]

get_recommendations(2, num_recommend = 20)


2      2
3      3
4      4
6      6
7      7
8      8
10    10
11    11
14    14
15    15
17    17
18    18
21    21
24    24
27    27
28    28
30    30
32    32
35    35
36    36
Name: video_id, dtype: int64

In [26]:
captions[captions['video_id'] == 36]

,video_id,manual_cover_text,caption,topic_tag,first_level_category_id,first_level_category_name,second_level_category_id,second_level_category_name,third_level_category_id,third_level_category_name,tokenized
36,36,UNKNOWN,#二次元 #我是吃货 #快手推送 #搞笑,"[二次元,快手推送,我是吃货,搞笑]",9,喜剧,136,喜剧段子,-124,UNKNOWN,[CLS] # 二 次 元 # 我 是 吃 货 # 快 手 推 送 # 搞 笑 [SEP]
